In [1]:
import pandas as pd
import glob
import numpy as np
import os
from config import ROOT, schema_map, engine, financial_mapping
from cleaners import FMPCleaner, Cleaner
from fetchers import YFinanceFetcher
from sqlalchemy import create_engine, text
import tradingeconomics as te

In [26]:
files = glob.glob(f"{ROOT}/data/processed/*.csv")
files

['/Users/tnguyen287/Documents/finance-db/data/processed/factors_quarterly.csv',
 '/Users/tnguyen287/Documents/finance-db/data/processed/USstock_universe_point_in_time.csv',
 '/Users/tnguyen287/Documents/finance-db/data/processed/asset_class_returns_monthly.csv',
 '/Users/tnguyen287/Documents/finance-db/data/processed/sector_returns_monthly.csv',
 '/Users/tnguyen287/Documents/finance-db/data/processed/factors_daily.csv',
 '/Users/tnguyen287/Documents/finance-db/data/processed/asset_class_returns_daily.csv',
 '/Users/tnguyen287/Documents/finance-db/data/processed/macro_daily.csv',
 '/Users/tnguyen287/Documents/finance-db/data/processed/sector_returns_daily.csv',
 '/Users/tnguyen287/Documents/finance-db/data/processed/macro_quarterly.csv',
 '/Users/tnguyen287/Documents/finance-db/data/processed/macro_monthly.csv',
 '/Users/tnguyen287/Documents/finance-db/data/processed/asset_class_returns_quarterly.csv',
 '/Users/tnguyen287/Documents/finance-db/data/processed/factors_monthly.csv',
 '/User

In [27]:
for file in ['/Users/tnguyen287/Documents/finance-db/data/processed/asset_class_returns_daily.csv',
             '/Users/tnguyen287/Documents/finance-db/data/processed/sector_returns_daily.csv',
             '/Users/tnguyen287/Documents/finance-db/data/processed/factors_daily.csv']:
    df = pd.read_csv(file)
    print(f"\n{file}")
    print(f"Shape: {df.shape}")
    print(f"Date range: {df['date'].min()} to {df['date'].max()}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"Nulls:\n{df.isnull().sum()}")


/Users/tnguyen287/Documents/finance-db/data/processed/asset_class_returns_daily.csv
Shape: (9683, 16)
Date range: 1993-01-29 to 2026-04-21
Columns: ['date', 'bitcoin', 'real_estate', 'us_agg_bonds', 'large_cap_growth', 'large_cap_value', 'small_cap', 'long_term_treasury', 'equal_weight_sp500', 'emerging_markets', 'short_term_treasury', 'intl_developed', 'gold', 'oil', 'us_large_cap', 'high_yield_bonds']
Nulls:
date                      0
bitcoin                5449
real_estate            4260
us_agg_bonds           4008
large_cap_growth       3171
large_cap_value        3171
small_cap              3171
long_term_treasury     3714
equal_weight_sp500     3904
emerging_markets       3892
short_term_treasury    3714
intl_developed         3486
gold                   4296
oil                    4645
us_large_cap           1321
high_yield_bonds       4896
dtype: int64

/Users/tnguyen287/Documents/finance-db/data/processed/sector_returns_daily.csv
Shape: (6873, 12)
Date range: 1998-12-22 to 

In [29]:
df = pd.read_csv(f"{ROOT}/data/processed/factors_daily_merged.csv")
print(f"\n{file}")
print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Columns: {df.columns.tolist()}")
print(f"Nulls:\n{df.isnull().sum()}")


/Users/tnguyen287/Documents/finance-db/data/processed/factors_daily.csv
Shape: (26440, 9)
Date range: 1927-01-03 to 2026-03-31
Columns: ['date', 'market_excess_return', 'size_factor', 'value_factor', 'profitability_factor', 'investment_factor', 'risk_free_rate', 'momentum_factor', 'quality_factor']
Nulls:
date                        0
market_excess_return    10670
size_factor             10670
value_factor            10670
profitability_factor    10670
investment_factor       10670
risk_free_rate          10670
momentum_factor             0
quality_factor           8759
dtype: int64


In [30]:
fac = pd.read_csv(f"{ROOT}/data/processed/factors_daily_merged.csv", parse_dates=['date'])

# Find where FF5 factors start
ff5_start = fac[fac['market_excess_return'].notna()]['date'].min()

# Find where QMJ starts
qmj_start = fac[fac['quality_factor'].notna()]['date'].min()

print(f"FF5 factors start: {ff5_start}")
print(f"QMJ starts: {qmj_start}")
print(f"Momentum starts: {fac[fac['momentum_factor'].notna()]['date'].min()}")

FF5 factors start: 1963-07-01 00:00:00
QMJ starts: 1957-07-01 00:00:00
Momentum starts: 1927-01-03 00:00:00


In [31]:
fac_1999 = fac[fac['date'] >= '1999-01-01']

print(fac_1999.isnull().sum())
print(f"Shape: {fac_1999.shape}")

date                      0
market_excess_return    278
size_factor             278
value_factor            278
profitability_factor    278
investment_factor       278
risk_free_rate          278
momentum_factor           0
quality_factor            0
dtype: int64
Shape: (7108, 9)


In [35]:
df = pd.read_sql('SELECT * from daily_prices Limit 500', engine)

In [37]:
print('Columns:', df.columns.tolist())
print()

print('Shape (sample):', df.shape)
print('Date range:', df['date'].min().date(), 'to', df['date'].max().date())
print('Unique tickers:', df['ticker'].nunique())
print()

print('Nulls:')
print(df.isnull().sum())
print()

print('Sample tickers:')
print(df['ticker'].unique()[:20])
print()

print('Sample rows:')
print(df.head(5))

Columns: ['date', 'ticker', 'close', 'high', 'low', 'open', 'volume', 'adj close']

Shape (sample): (500, 8)
Date range: 1990-01-02 to 1990-01-02
Unique tickers: 500

Nulls:
date           0
ticker         0
close        470
high         470
low          470
open         470
volume       470
adj close    500
dtype: int64

Sample tickers:
<ArrowStringArray>
['0P00000SXJ',          'A',         'AA',       'AABB',      'AABPX',
      'AABVF',      'AACAF',      'AACAY',       'AACB',      'AACBR',
      'AACBU',       'AACG',      'AACIU',       'AACS',      'AACTF',
      'AADBX',      'AAFRF',       'AAGC',      'AAGFF',       'AAGH']
Length: 20, dtype: str

Sample rows:
        date      ticker      close       high        low       open  \
0 1990-01-02  0P00000SXJ        NaN        NaN        NaN        NaN   
1 1990-01-02           A        NaN        NaN        NaN        NaN   
2 1990-01-02          AA  12.796237  12.796237  12.563578  12.690484   
3 1990-01-02        AABB        